# Exercise 2.

In [ ]:
# --- Read the BLOSUM62 matrix from .txt file ---

# Read and store the whole blosum62.txt file as a list of strings, each string representing a row

f = open("blosum62.txt", "r")
full_file_blosum62 = f.readlines()
f.close()

In [ ]:
# --- Separate the residues ---

# Only looking at second line containing the residues
# Filter out the white space and newline at the end, store each letter as individual string

residues = full_file_blosum62[1].split()


# --- Store the matrix values as nested list (list of lists) --- 

# Each row of values in the matrix is stored as a list of value

score_matrix = []

for row_str in full_file_blosum62[2:]:
    
    # Turn row of string into list of string elements
    row_list = row_str.split()

    row_value_list = []

    for elem in row_list[1:]:
        # Skipping first element which is a residue letter

        row_value_list.append(int(elem))

    score_matrix.append(row_value_list)


# --- Turn into dictionary ---

# Since the list of residues have indices corresponding to the position in the scoring matrix we can easily construct a dictionary with
# pairs of residues (as tuple of two string) and their score

blosum62_score_matrix_dict = {}

for i, resA in enumerate(residues):
    for j, resB in enumerate(residues):
        blosum62_score_matrix_dict[(resA, resB)] = score_matrix[i][j]

In [18]:
def faa_fasta_reader(full_filename):
    """ Return sequence as string of residues """

    f = open(full_filename, "r")
    full_file = f.readlines()
    f.close()

    seq = ""

    # Skip first line in file
    for row_string in full_file[1:]:
        for res in row_string:
            if res == " " or res == "\n":
                pass

            else:
                seq += res

    return seq

In [66]:
seqA = faa_fasta_reader("sequence_Q9SIE0_2.fasta")[:5]
seqB = faa_fasta_reader("sequence_Q9SIE0_2.fasta")[:5]
print(seqA)
print(seqB)

MTNPL
MTNPL


In [78]:
def basic_matrix_setup(seqA, seqB):

    scoring_matrix = []

    n_rows = len(seqB) + 2 
    n_columns = len(seqA) + 2

    # Fill entire matrix with zero, to create correct len if list
    for i in range(n_rows):

        row_list = []

        for j in range(n_columns):

            row_list.append(0)

        scoring_matrix.append(row_list)

    # Fill in the two first rows and columns
    scoring_matrix[0][0] = " "
    scoring_matrix[0][1] = "-"
    scoring_matrix[1][0] = "-"

    for i in range(len(seqA)):
        scoring_matrix[0][i+2] = seqA[i]

    for j in range(len(seqB)):
        scoring_matrix[j+2][0] = seqB[j]


    return scoring_matrix

In [68]:
scoring_matrix = basic_matrix_setup(seqA, seqB)

for row in scoring_matrix:
    print(row)

[' ', '-', 'M', 'T', 'N', 'P', 'L']
['-', 0, 0, 0, 0, 0, 0]
['M', 0, 0, 0, 0, 0, 0]
['T', 0, 0, 0, 0, 0, 0]
['N', 0, 0, 0, 0, 0, 0]
['P', 0, 0, 0, 0, 0, 0]
['L', 0, 0, 0, 0, 0, 0]


In [69]:
resA = "W"
resB = "W"

blosum62_score_matrix_dict.get((resA, resB))

11

In [70]:
for i in range(2, len(seqA) + 2):
    for j in range(2, len(seqB) + 2):
        resA = scoring_matrix[0][j]
        resB = scoring_matrix[i][0]
        # print(resA, resB)
        # resA = seqA[j]
        # resB = seqB[i]
        scoring_matrix[i][j] = blosum62_score_matrix_dict.get((resA, resB))

for row in scoring_matrix:
    print(row)

[' ', '-', 'M', 'T', 'N', 'P', 'L']
['-', 0, 0, 0, 0, 0, 0]
['M', 0, 5, -1, -2, -2, 2]
['T', 0, -1, 5, 0, -1, -1]
['N', 0, -2, 0, 6, -2, -3]
['P', 0, -2, -1, -2, 7, -3]
['L', 0, 2, -1, -3, -3, 4]


In [79]:
seqA = "AAGT"
seqB = "AT"

In [184]:
def affine_gap_function():
    return

def initial_matrix_setup(len_seqA, len_seqB, gap_open_pen, gap_extend_pen):

    if gap_open_pen > 0 or gap_extend_pen > 0:
        raise ValueError("Penalties must strictly be negative")

    h = gap_open_pen
    s = gap_extend_pen

    V_matrix = []
    E_matrix = []
    F_matrix = []
    G_matrix = []

    n_rows = len_seqB + 1
    n_columns = len_seqA + 1

    for i in range(n_rows):
        row_list_V = []
        row_list_E = []
        row_list_F = []
        row_list_G = []

        for j in range(n_columns):
            row_list_V.append(0)
            row_list_E.append(0)
            row_list_F.append(0)
            row_list_G.append(0)

        V_matrix.append(row_list_V)
        E_matrix.append(row_list_E)
        F_matrix.append(row_list_F)
        G_matrix.append(row_list_G)

    V_matrix[0][0] = 0
    E_matrix[0][0] = 0
    F_matrix[0][0] = 0
    G_matrix[0][0] = 0

    # V matrix
    for i in range(1, n_rows):
        V_matrix[i][0] = h + i*s

    for j in range(1, n_columns):
        V_matrix[0][j] = h + j*s

    # E matrix
    for i in range(1, n_rows):
        E_matrix[i][0] = 2*h + (1+i)*s

    for j in range(1, n_columns):
        E_matrix[0][j] = -99

    # F matrix
    for i in range(1, n_rows):
        F_matrix[i][0] = -99

    for j in range(1, n_columns):
        F_matrix[0][j] = 2*h + (j+1)*s
    
    # G matrix
    for i in range(1, n_rows):
        G_matrix[i][0] = h + i*s

    for j in range(1, n_columns):
        G_matrix[0][j] = h + j*s
    
    return V_matrix, E_matrix, F_matrix, G_matrix

In [185]:
h = -1 # opening pen
s = -1 # extend pen

In [186]:
V_matrix, E_matrix, F_matrix, G_matrix = initial_matrix_setup(len(seqA), len(seqB), gap_open_pen=h, gap_extend_pen=s)

for row in V_matrix:
    print(row)

for row in E_matrix:
    print(row)

for row in F_matrix:
    print(row)

for row in G_matrix:
    print(row)

[0, -2, -3, -4, -5]
[-2, 0, 0, 0, 0]
[-3, 0, 0, 0, 0]
[0, -99, -99, -99, -99]
[-4, 0, 0, 0, 0]
[-5, 0, 0, 0, 0]
[0, -4, -5, -6, -7]
[-99, 0, 0, 0, 0]
[-99, 0, 0, 0, 0]
[0, -2, -3, -4, -5]
[-2, 0, 0, 0, 0]
[-3, 0, 0, 0, 0]
